# Proyecto Aula - Tercer Corte
## Limpieza de Datos con Programacion Orientada a Objetos

**Programacion de Computadores**
Universidad ECCI - Facultad de Ingenieria - 2026-I

---

### Contexto

Eres parte del equipo de analisis de datos de una consultora. Te entregan
los resultados de una encuesta socioeconomica de 50 personas, pero el
archivo llego **sucio**: tiene errores de formato, inconsistencias y
registros invalidos.

Antes de poder analizar los datos, debes **limpiarlos**. Tu tarea es
construir una clase `Limpiador` que, usando metodos, corrija cada tipo de
error y finalmente produzca un resumen estadistico correcto.

### Como sabras que limpiaste bien

Te entregamos el **analisis descriptivo esperado** (los valores correctos).
Si tu clase limpia bien los datos, tu metodo `resumen()` debe producir
EXACTAMENTE esos valores. Esa es tu verificacion.

---
## Paso 0: Cargar el archivo de datos

Ejecuta la siguiente celda y sube el archivo `encuesta_sucia.csv` que te
entrego el profesor.

In [2]:
from google.colab import files
subidos = files.upload()

Saving encuesta_sucia.csv to encuesta_sucia.csv


In [3]:
# Vista rapida del contenido crudo (primeras 12 lineas)
archivo = open("encuesta_sucia.csv", encoding="utf-8")
contador = 0
for linea in archivo:
    print(linea.strip())
    contador = contador + 1
    if contador == 12:
        break
archivo.close()

id,ciudad,edad,ingreso,genero
36,medellin ,41 años,2.200.000,FEMENINO
15,Bogotá,42 años,3.500.000,Masculino
33,Medellín,47 años,1300000,m
5,Cartagena, 30 ,8000000,masculino
1,  Bogota , 19 ,3.500.000 COP,FEMENINO
35,CALI,58 años,$ 8.000.000, F
29,Medellín,61 anios,5000000,Femenino
19, medellin,47 años,5.000.000,masculino
27,medellin ,26 anios,$ 8.000.000, M
25,Medellín,65,8000000,M
37,cartagena, 38 ,$6.500.000,F


---
## El analisis descriptivo esperado (la "verdad")

Si limpias correctamente, tu `resumen()` debe producir estos valores:

| Indicador | Valor correcto |
|---|---|
| Numero de registros validos (n) | 50 |
| Ciudades unicas | Barranquilla, Bogota, Cali, Cartagena, Medellin (5) |
| Conteo por ciudad | Barranquilla 5, Bogota 10, Cali 7, Cartagena 13, Medellin 15 |
| Conteo por genero | F = 29, M = 21 |
| Edad minima / maxima | 18 / 65 |
| Edad promedio | 39.16 |
| Ingreso minimo / maximo | 1300000 / 8000000 |
| Ingreso promedio | 4018000.0 |

---
## Tu mision: diagnostico de errores

El archivo tiene **varias categorias de error combinadas**. Tu trabajo es
inspeccionar los datos, identificar los casos concretos de cada categoria y
escribir los metodos que los corrijan. Estas son las categorias presentes:

- **Texto desordenado:** espacios sobrantes, mayusculas/minusculas mezcladas
  y tildes inconsistentes en `ciudad`.
- **Genero inconsistente:** aparece como `f`, `F`, `Femenino`, `m`, `Masculino`, etc.
- **Ingreso con formato:** viene como texto con simbolos: `$2.200.000`,
  `3.500.000 COP`, `5000000`.
- **Edad con ruido:** trae texto adicional: `39 anios`, `61 años`, ` 45 `.
- **Registros duplicados:** algunas personas (mismo `id`) aparecen mas de una vez.
- **Registros invalidos:** filas con ciudad inexistente, edad imposible,
  ingreso no numerico o campos vacios. Estas filas se **descartan completas**.

> Pista: la `ciudad` valida solo puede ser una de las cinco de la tabla
> esperada. El `genero` valido solo puede ser `F` o `M`. La edad valida esta
> entre 0 y 120.

---
# PARTE A - Ejemplos resueltos (estudialos)

Aqui tienes el inicio de la clase `Limpiador` con **dos metodos ya
resueltos** como ejemplo del patron que debes seguir: `limpiar_texto()` y
`limpiar_genero()`. Estudia como funcionan: reciben un valor sucio y
devuelven el valor limpio.

In [4]:
import csv

class Limpiador:
    def __init__(self, ruta_csv):
        # Carga las filas del CSV (sin el encabezado) en una lista de listas
        self.crudo = []
        archivo = open(ruta_csv, encoding="utf-8")
        lector = csv.reader(archivo)
        encabezado = next(lector)
        for fila in lector:
            self.crudo.append(fila)
        archivo.close()
        self.limpio = []  # aqui guardaras los datos ya limpios

    # ----- EJEMPLO RESUELTO 1: limpiar texto de ciudad -----
    def limpiar_texto(self, valor):
        # Quita espacios, pasa a minuscula, quita tildes y capitaliza
        v = valor.strip()
        v = v.lower()
        v = v.replace("a" + chr(769), "a")  # por si hay tildes combinadas
        v = v.replace(chr(225), "a")  # a con tilde
        v = v.replace(chr(233), "e")  # e con tilde
        v = v.replace(chr(237), "i")  # i con tilde
        v = v.replace(chr(243), "o")  # o con tilde
        v = v.replace(chr(250), "u")  # u con tilde
        if len(v) > 0:
            v = v[0].upper() + v[1:]
        return v

    # ----- EJEMPLO RESUELTO 2: limpiar genero -----
    def limpiar_genero(self, valor):
        v = valor.strip()
        v = v.lower()
        if v == "f" or v == "femenino":
            return "F"
        if v == "m" or v == "masculino":
            return "M"
        return ""  # valor invalido, no reconocido

In [5]:
# Prueba de los ejemplos resueltos
prueba = Limpiador("encuesta_sucia.csv")
print(prueba.limpiar_texto("  MEDELLIN "))    # debe dar: Medellin
print(prueba.limpiar_texto("Bogot" + chr(225)))  # debe dar: Bogota
print(prueba.limpiar_genero("Femenino"))      # debe dar: F
print(prueba.limpiar_genero(" M "))           # debe dar: M

Medellin
Bogota
F
M


---
# PARTE B - Tu trabajo (completa los metodos)

Ahora te toca a ti. Vas a **agregar** los metodos que faltan a la clase.
Para mantener todo junto, copiamos la clase completa abajo: los dos metodos
de ejemplo ya estan, y tu debes completar los que tienen `# COMPLETAR`.

Lee con cuidado los comentarios de cada metodo.

In [11]:
import csv
from collections import Counter

class Limpiador:
    def __init__(self, ruta_csv):
        self.crudo = []
        archivo = open(ruta_csv, encoding="utf-8")
        lector = csv.reader(archivo)
        encabezado = next(lector)
        for fila in lector:
            self.crudo.append(fila)
        archivo.close()
        self.limpio = []

    # ===== YA RESUELTO =====
    def limpiar_texto(self, valor):
        v = valor.strip()
        v = v.lower()
        v = v.replace(chr(225), "a")
        v = v.replace(chr(233), "e")
        v = v.replace(chr(237), "i")
        v = v.replace(chr(243), "o")
        v = v.replace(chr(250), "u")
        if len(v) > 0:
            v = v[0].upper() + v[1:]
        return v

    # ===== YA RESUELTO =====
    def limpiar_genero(self, valor):
        v = valor.strip()
        v = v.lower()
        if v == "f" or v == "femenino":
            return "F"
        if v == "m" or v == "masculino":
            return "M"
        return ""

    # ===== COMPLETAR 1 =====
    def limpiar_ingreso(self, valor):
        # Convierte el texto del ingreso a un numero entero, o None si no es valido.
        v = str(valor).strip()
        v = v.replace("$", "").replace(".", "").replace("COP", "").strip()

        try:
          return int(v)
        except:
          return None



    # ===== COMPLETAR 2 =====
    def limpiar_edad(self, valor):
        # Convierte el texto de la edad a un numero entero, o None si no es valido.
        v = str(valor).strip()
        v = v.lower()
        v = v.replace("años", "").replace("anios", "").strip()
        try:
            return int(v)
        except ValueError:
            return None

    # ===== COMPLETAR 3 =====
    def es_valida(self, ciudad, edad, ingreso, genero):
        # Devuelve True si el registro es valido, False si debe descartarse.
        # Pista: la ciudad solo puede ser una de estas cinco; el genero solo "F" o "M";
        # la edad debe estar entre 0 y 120.
        ciudades_ok = ["Bogota", "Medellin", "Cali", "Barranquilla", "Cartagena"]
        if ciudad not in ciudades_ok:
            return False
        if genero not in ["F", "M"]:
            return False
        if not (isinstance(edad, int) and 0 <= edad <= 120):
            return False
        if not (isinstance(ingreso, int) and ingreso > 0):
            return False

        return True

    # ===== COMPLETAR 4 =====
    def limpiar(self):
        # Recorre los datos crudos, limpia cada campo, descarta los registros
        # invalidos y los duplicados, y guarda el resultado en self.limpio.
        self.limpio = []
        seen_ids = set()

        for fila in self.crudo:
            raw_id, raw_ciudad, raw_edad, raw_ingreso, raw_genero = fila

            _id = None
            try:
                _id = int(raw_id.strip())
            except ValueError:
                continue

            ciudad = self.limpiar_texto(raw_ciudad)
            edad = self.limpiar_edad(raw_edad)
            ingreso = self.limpiar_ingreso(raw_ingreso)
            genero = self.limpiar_genero(raw_genero)

            if self.es_valida(ciudad, edad, ingreso, genero) and _id not in seen_ids:
                self.limpio.append({
                    'id': _id,
                    'ciudad': ciudad,
                    'edad': edad,
                    'ingreso': ingreso,
                    'genero': genero
                })
                seen_ids.add(_id)

    # ===== COMPLETAR 5 =====
    def resumen(self):

        # Calcula y muestra el analisis descriptivo de self.limpio (n, ciudades,
        # conteos, minimos, maximos y promedios de edad e ingreso).
        n = len(self.limpio)
        print(f"Numero de registros validos (n): {n}")

        if n == 0:
            print("No hay datos limpios para generar un resumen.")
            return

        ciudades = [record['ciudad'] for record in self.limpio]
        edades = [record['edad'] for record in self.limpio]
        ingresos = [record['ingreso'] for record in self.limpio]
        generos = [record['genero'] for record in self.limpio]

        # Ciudades unicas
        ciudades_unicas = sorted(list(set(ciudades)))
        print(f"Ciudades unicas: {', '.join(ciudades_unicas)} ({len(ciudades_unicas)})")

        # Conteo por ciudad
        conteo_ciudad = Counter(ciudades)
        conteo_ciudad_str = ', '.join([f"{k} {v}" for k, v in sorted(conteo_ciudad.items())])
        print(f"Conteo por ciudad: {conteo_ciudad_str}")

        # Conteo por genero
        conteo_genero = Counter(generos)
        conteo_genero_str = f"F = {conteo_genero.get('F', 0)}, M = {conteo_genero.get('M', 0)}"
        print(f"Conteo por genero: {conteo_genero_str}")

        # Edad
        min_edad = min(edades)
        max_edad = max(edades)
        promedio_edad = sum(edades) / n
        print(f"Edad minima / maxima: {min_edad} / {max_edad}")
        print(f"Edad promedio: {promedio_edad:.2f}")

         # Ingreso
        min_ingreso = min(ingresos)
        max_ingreso = max(ingresos)
        promedio_ingreso = sum(ingresos) / n
        print(f"Ingreso minimo / maximo: {min_ingreso} / {max_ingreso}")
        print(f"Ingreso promedio: {promedio_ingreso:.1f}")

---
## Paso final: ejecuta y verifica

Cuando completes todos los metodos, ejecuta la siguiente celda. Los valores
que imprime tu `resumen()` deben coincidir con la tabla de la "verdad".

In [13]:
limpiador = Limpiador("encuesta_sucia.csv")
limpiador.limpiar()
print("Registros validos tras limpiar:", len(limpiador.limpio))
print("-" * 40)
limpiador.resumen()

Registros validos tras limpiar: 50
----------------------------------------
Numero de registros validos (n): 50
Ciudades unicas: Barranquilla, Bogota, Cali, Cartagena, Medellin (5)
Conteo por ciudad: Barranquilla 5, Bogota 10, Cali 7, Cartagena 13, Medellin 15
Conteo por genero: F = 29, M = 21
Edad minima / maxima: 18 / 65
Edad promedio: 39.16
Ingreso minimo / maximo: 1300000 / 8000000
Ingreso promedio: 4018000.0


---
## Entrega

1. Completa todos los metodos marcados con `# COMPLETAR`.
2. Verifica que tu `resumen()` reproduce la tabla esperada.
3. Responde en la celda de abajo (texto): ¿que tipo de error te parecio
   mas dificil de corregir y por que?

**Tu respuesta:**

(NameError: name 'Counter' is not defined. Aunque no es un error de lógica de limpieza de datos en sí, fue una dificultad técnica clave porque impedía que el método resumen() funcionara en absoluto, lo que a su vez bloqueaba la verificación del resto de la limpieza)